# Tool 7 — Manually reject flagged epochs — Curry twin (Phase 2b)

*Curry twin: functionally identical to the EDF tool 7. Tool 7 reads only tool-6 `*_all-epo.fif` +
its optional companions, which the Curry tool 6 writes in the same MNE/TSV format, so no analysis
code differs — only the shared-library import path. The high-density support (32–64 channel
montages) lives in the shared `tools/qc_rejected_epochs_lib.py` and adapts to the channel count.
Generated by `tools_curry/_make_tool7_curry.py` — do not hand-edit; edit the EDF notebook and
re-run the generator.*

Inspect the epochs that **6_preprocessing** flagged, decide **keep / reject** for each, and export a
validated `*_clean-epo.fif`. Reads one participant's `*_all-epo.fif` (+ optional
`*_preprocessing_params.json`) from the selected *raw-epochs* folder — the raw `.cdt` is never reloaded (EEG-only,
whatever channels the `.fif` holds).

**Workflow:** ① Load a participant (+ optional **channel triage**) → ② Global per-stage report →
③ Navigate rejected epochs → ④ Override decisions & save. Tool-6 outputs are never modified.

*High-density montages (32–64 channels) are supported: the montage, the metric table and the 1/f
cost all adapt to the channel count, and Section 1 lets you drop globally-bad channels and switch
the epoch rule away from “any flagged channel”. On a 3–6 channel PSG montage nothing changes.*

In [ ]:
try:
    import os, sys, json, io
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from ipyfilechooser import FileChooser

    # Make the shared library importable whether Voila is launched from the repo root (lib in
    # tools/) or from tools_curry/ (this twin lives there; the shared lib stays in ../tools).
    _here = os.getcwd()
    _lib_candidates = [_here, os.path.join(_here, 'tools'),
                       os.path.join(os.path.dirname(_here), 'tools'),
                       os.path.join(_here, '..', 'tools')]
    for _cand in _lib_candidates:
        if os.path.isfile(os.path.join(_cand, 'qc_rejected_epochs_lib.py')):
            _cand = os.path.abspath(_cand)
            if _cand not in sys.path:
                sys.path.insert(0, _cand)
            break
    import qc_rejected_epochs_lib as L
except ImportError as e:
    print("⚠️ Import error — a package is missing:", e)
else:
    print("✅ Packages imported successfully!")

# Shared mutable state across sections.
S = {}

# Output areas (one per section).
out_load   = widgets.Output()
out_report = widgets.Output()
out_nav    = widgets.Output()
out_review = widgets.Output()

def show_fig(fig, close=True):
    # Render a matplotlib Figure as a PNG widget — reliable under Voila with the Agg backend
    # (plain display(fig) would only print the figure's text repr). Same pattern as tools 5/8.
    # close=False keeps the Figure open so it can be reused (e.g. saved into an mne.Report afterwards).
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    if close:
        plt.close(fig)
    display(widgets.Image(value=buf.getvalue(), format='png'))

if not L.HAS_SPECPARAM:
    print('WARNING: specparam not available — 1/f metrics will be blank.')


## Section 1 — Load a participant

In [ ]:
# ---- Section 1: folder pickers + participant + selection (stages / methods / event types / channels) ----
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Data folder</b> (holds <code>derivatives/</code> + <code>reports_preprocessing/</code>) '
                 '&mdash; optional: pre-points the raw folder and anchors the reports output:')

fc_raw = FileChooser(os.getcwd())
fc_raw.show_only_dirs = True
fc_raw.title = ('<b>Raw-epochs folder</b> (holds the tool-6 <code>*_all-epo.fif</code>, e.g. '
                '<code>derivatives/raw_epo</code>):')

# OPTIONAL third picker: the tool-6 reports folder, read ONLY for {file_id}_epoch_channel_rejection.tsv
# (the per-(epoch, channel) flags). It is what makes per-channel badness exact — it carries the
# per-channel 1/f flags, which tool 7 would otherwise have to refit (~8 min on a 32-channel night).
# Leave it unset and the channel view falls back to time-domain flags recomputed from the signal.
fc_reports = FileChooser(os.getcwd())
fc_reports.show_only_dirs = True
fc_reports.title = ('<b>Reports folder</b> (holds the tool-6 <code>*_epoch_channel_rejection.tsv</code>, e.g. '
                    '<code>reports_preprocessing</code>) &mdash; <i>optional</i>: exact per-channel flags:')

dd_part    = widgets.Dropdown(description='Participant:', options=[], style={'description_width':'110px'},
                              layout=widgets.Layout(width='340px'))
txt_custom = widgets.Text(description='Custom stages:', value='', style={'description_width':'110px'},
                          layout=widgets.Layout(width='340px'),
                          placeholder='e.g. N4  (comma-separated, optional)')
lbl_scan   = widgets.HTML('<i>Select the raw-epochs folder (optionally set the data folder first).</i>')
lbl_done   = widgets.HTML('')     # 'already processed' badge for the selected participant (informative only)
btn_load   = widgets.Button(description='Load participant', button_style='primary', icon='folder-open')

# --- Selection: which stages / methods / event types define a 'rejected' epoch (a la tool 7bis) ---
# Stages + methods are picked BEFORE load (they also skip the slow PSD/1-f work on out-of-scope stages);
# the event-type sub-boxes are data-dependent so they populate AT load (all ticked). Change a tick then
# click 'Apply selection' to recompute the rejected-epoch set (this resets any manual overrides).
_mstyle     = {'description_width':'initial'}
_cblay      = widgets.Layout(width='auto', margin='2px 12px 2px 0')
box_stages  = widgets.HBox([])
_stage_cb   = {}
_method_cb  = {m: widgets.Checkbox(value=True, description=L.METHOD_LABEL[m], indent=False,
                                   style=_mstyle, layout=_cblay) for m in L.METHOD_ORDER}
box_methods = widgets.HBox([widgets.HTML('<b>Methods:</b>')] + [_method_cb[m] for m in L.METHOD_ORDER])
box_evt     = widgets.Box([widgets.HTML('<small style="color:#888;">event types appear here after load</small>')],
                          layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
_evt_cb     = {}
# Reveal the event-type box from the 'event' method checkbox (initial display derived from its value).
box_evt.layout.display = '' if _method_cb['event'].value else 'none'
def _toggle_evt(change):
    box_evt.layout.display = '' if change['new'] else 'none'
_method_cb['event'].observe(_toggle_evt, names='value')
btn_apply   = widgets.Button(description='Apply selection', icon='sync', disabled=True,
                             tooltip='Recompute the rejected-epoch set from the ticked stages / methods / '
                                     'event types / channels (resets manual overrides)')

# --- Channel triage (matters on a HIGH-DENSITY montage) -------------------------------------------
# Tool 6 does not flag epochs, it flags (epoch x channel) PAIRS. Two questions follow, in this order:
#   1. which CHANNELS do I keep? A channel flagged on most of the night is a bad electrode: drop it
#      (it then flags no epoch at all AND leaves the saved clean-epo). Same rule as tool 7bis.
#   2. which EPOCHS do I reject, among the KEPT channels? 'any flagged channel' (the classic rule), or
#      only when more than P% of them are flagged (tool 7bis's rule).
# The order matters: a dropped channel's flags no longer count towards the epoch decision.
# Why this exists: with 3 channels 'any flagged channel' is sound; on a real 32-channel Curry night it
# rejects 72.6% of the epochs, a third of the file on a SINGLE electrode out of 32 - where the right
# action is to drop that electrode, not 30 s of EEG.
# Both default to the classic behaviour (drop nothing, rule 'any'), so out of the box the decision is
# exactly tool 6's and a sparse PSG montage is completely unaffected.
DROP_PCT_SUGGESTED = 20.0        # only used to word the load-time hint (tool 7bis's default value)
_chan_cb   = {}
box_chan   = widgets.Box([widgets.HTML('<small style="color:#888;">channels appear here after load</small>')],
                         layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%',
                                               max_height='170px', overflow_y='auto',
                                               border='1px solid #ddd', padding='4px'))
ft_drop_pct = widgets.FloatText(description='Drop channels flagged in more than (%) of the epochs:',
                                value=0.0, step=5, style={'description_width':'330px'},
                                layout=widgets.Layout(width='410px'))
lbl_drop_hint = widgets.HTML(
    '<small>0 = keep every channel (default &mdash; the decision then matches tool 6 exactly). Set it '
    '(tool 7bis uses 20) to untick the channels above it; you can re-tick any of them by hand '
    'afterwards, but changing the value re-applies the rule and discards those corrections. An '
    'unticked channel flags no epoch and is removed from the saved clean-epo.</small>')
lbl_badness = widgets.HTML('')
dd_rule = widgets.Dropdown(description='Epoch rule:', value='any',
                           options=[('any flagged channel (classic)', 'any'),
                                    ('> P % of the kept channels', 'pct')],
                           style={'description_width':'80px'}, layout=widgets.Layout(width='330px'))
ft_rule_value = widgets.FloatText(description='P (%)', value=20.0, step=5,
                                  style={'description_width':'50px'}, layout=widgets.Layout(width='130px'))
# Dropdown-revealed widget: the initial display MUST follow the current value, not a hard-coded 'none'
# (an observer only fires on a *change*) - see SPEC 'Checkbox-revealed parameter widgets'.
ft_rule_value.layout.display = '' if dd_rule.value != 'any' else 'none'
def _toggle_rule(change):
    ft_rule_value.layout.display = '' if change['new'] != 'any' else 'none'
dd_rule.observe(_toggle_rule, names='value')
box_triage = widgets.VBox([
    widgets.HTML('<b>Channels kept</b> &mdash; tool 6 flags each <i>(epoch &times; channel)</i> pair. '
                 'First drop the electrodes that are bad over the whole night, then choose how many '
                 'flagged channels it takes to reject an epoch:'),
    ft_drop_pct, lbl_drop_hint,
    box_chan, lbl_badness,
    widgets.HBox([dd_rule, ft_rule_value]),
])

def _build_stage_boxes():
    _stage_cb.clear()
    stages = ['W','N1','N2','N3','R'] + [s for s in L.parse_custom_field(txt_custom.value)
                                         if s not in ('W','N1','N2','N3','R')]
    for st in stages:
        _stage_cb[st] = widgets.Checkbox(value=True, description=st, indent=False,
                                         style=_mstyle, layout=widgets.Layout(width='auto', margin='2px 10px 2px 0'))
    box_stages.children = [widgets.HTML('<b>Stages:</b>')] + [_stage_cb[s] for s in stages]

_build_stage_boxes()
txt_custom.observe(lambda chg: _build_stage_boxes(), 'value')

def _selected_stages():
    return [s for s, cb in _stage_cb.items() if cb.value]
def _selected_methods():
    return [m for m in L.METHOD_ORDER if _method_cb[m].value]
def _selected_event_types():
    return [t for t, cb in _evt_cb.items() if cb.value]
def _dropped_channels():
    return [c for c, cb in _chan_cb.items() if not cb.value]
def _epoch_rule():
    return dd_rule.value, float(ft_rule_value.value)

def _populate_event_types(evt_types):
    _evt_cb.clear()
    if not evt_types:
        box_evt.children = [widgets.HTML('<small style="color:#888;">no event types in this file '
                                         '(tool-6 event flagging did not run)</small>')]
        return
    for t in evt_types:
        _evt_cb[t] = widgets.Checkbox(value=True, description=t, indent=False, style=_mstyle, layout=_cblay)
    box_evt.children = [widgets.HTML('<b>Event types:</b>')] + [_evt_cb[t] for t in evt_types]

def _populate_channels(ch_names):
    _chan_cb.clear()
    for c in ch_names:
        _chan_cb[c] = widgets.Checkbox(value=True, description=c, indent=False, style=_mstyle,
                                       layout=widgets.Layout(width='auto', margin='1px 10px 1px 0'))
    box_chan.children = list(_chan_cb.values())

def _refresh_channel_labels():
    # Per-channel badness (% of the in-scope epochs a selected method flags the channel in) shown right
    # in the checkbox label, so "why is this one unticked?" reads without a second lookup. The dict KEY
    # stays the plain channel name - only the description changes.
    bad = S.get('badness')
    for i, (c, cb) in enumerate(_chan_cb.items()):
        cb.description = c if bad is None else f'{c} — {bad[i]:.0f}%'

def _refresh_badness_label():
    # Provenance of the per-channel flags (the badness itself is in each channel's label above).
    src = S.get('flags_source', '')
    if not src:
        lbl_badness.value = ''
        return
    warn = ('' if 'reports' in src else
            ' <span style="color:#e65100;">&#9888; time-domain methods only &mdash; select the reports '
            'folder for exact per-channel 1/f flags</span>')
    lbl_badness.value = f'<small>Channel flags: {src}{warn}</small>'

def _output_folders(folder):
    # clean .fif beside the raw folder; reports beside reports_preprocessing/ (subtree relative to raw).
    raw_root = Path(S['raw_root']); data_root = Path(S['data_root'])
    try:
        subtree = Path(folder).relative_to(raw_root)
    except Exception:
        subtree = Path('.')
    return (raw_root.parent / 'clean_epo_manual' / subtree,
            data_root / 'reports_rejection_manual' / subtree)

def _is_processed(fid, folder):
    # 'done' = both the clean-epo AND a review/decision record on disk (skip convention).
    out_folder, reports_folder = _output_folders(folder)
    fif_ok = (out_folder / f'{fid}_clean-epo.fif').exists()
    tsv_ok = ((reports_folder / f'{fid}_epoch_rejection_reviewed.tsv').exists()
              or (reports_folder / f'{fid}_manualreject_decision.tsv').exists())
    return fif_ok and tsv_ok

def _update_done_badge(*_):
    fid = dd_part.value
    if not fid or 'parts' not in S or fid not in S['parts']:
        lbl_done.value = ''
        return
    try:
        if _is_processed(fid, S['parts'][fid]['folder']):
            lbl_done.value = ('<span style="color:#2e7d32;">&#10003; already processed in manual '
                              '(clean-epo + review found)</span>')
        else:
            lbl_done.value = '<span style="color:#888;">&mdash; not yet processed in manual</span>'
    except Exception:
        lbl_done.value = ''

def _derive_data_root(raw_root):
    if fc_data.selected_path:
        return Path(fc_data.selected_path)
    rr = Path(raw_root)
    for anc in [rr] + list(rr.parents):
        if anc.name == 'derivatives':
            return anc.parent
    return rr.parent

def _on_data(chooser):
    try:
        data = fc_data.selected_path
        if not data:
            return
        start = Path(data) / 'derivatives' if (Path(data) / 'derivatives').is_dir() else Path(data)
        fc_raw.reset(path=str(start))
        fc_raw.title = ('<b>Raw-epochs folder</b> (holds the tool-6 <code>*_all-epo.fif</code>, e.g. '
                        '<code>derivatives/raw_epo</code>):')
        fc_reports.reset(path=str(data))
        fc_reports.title = ('<b>Reports folder</b> (holds the tool-6 <code>*_epoch_channel_rejection.tsv</code>, '
                            'e.g. <code>reports_preprocessing</code>) &mdash; <i>optional</i>: '
                            'exact per-channel flags:')
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Data folder error: {e}</span>'

def _on_raw(chooser):
    try:
        raw = fc_raw.selected_path
        if not raw:
            return
        raw_root = Path(raw)
        parts = L.find_participants(raw_root)
        S['parts'] = {p['file_id']: p for p in parts}
        S['raw_root'] = raw_root
        S['data_root'] = _derive_data_root(raw_root)
        dd_part.options = [p['file_id'] for p in parts]
        cs = L.load_custom_stages(S['data_root'])
        txt_custom.value = ','.join(cs)
        _build_stage_boxes()
        n_done = sum(1 for p in parts if _is_processed(p['file_id'], p['folder']))
        colour = '#2e7d32' if parts else '#c62828'
        lbl_scan.value = (f'<span style="color:{colour}">{len(parts)} participant(s) in the raw folder '
                          f'&mdash; {n_done} already processed in manual.</span>'
                          f'<br><small>clean .fif &rarr; {raw_root.parent / "clean_epo_manual"}'
                          f'<br>reports &rarr; {S["data_root"] / "reports_rejection_manual"}</small>')
        _update_done_badge()
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Scan error: {e}</span>'

fc_data.register_callback(_on_data)
fc_raw.register_callback(_on_raw)
dd_part.observe(_update_done_badge, 'value')

def _find_channel_flags_tsv(fid):
    # Locate {fid}_epoch_channel_rejection.tsv WITHIN the chosen reports folder only (never a recursive
    # scan of derivatives/, so versioned tool-6 runs never mix). Returns the folder or None.
    root = fc_reports.selected_path
    if not root:
        return None
    try:
        for f in Path(root).rglob(f'{fid}_epoch_channel_rejection.tsv'):
            return f.parent
    except Exception:
        pass
    return None

def _load_channel_flags(fid, P, thr):
    # Per-(epoch, channel) flags: exact from the tool-6 TSV when the reports folder is set, else
    # recomputed from the signal (time-domain methods only — no per-channel 1/f).
    n = len(P['stages'])
    folder = _find_channel_flags_tsv(fid)
    if folder is not None:
        pf = L.load_channel_flags(folder, fid, P['ch_names'], n)
        if pf:
            return pf, f'reports TSV ({folder.name})'
    return L.recompute_channel_flags(P, thr), 'recomputed from signal (time-domain only)'

def _recompute_reject(reset_overrides=True):
    # Recompute the rejected-epoch set from the ticked methods / stages / event types / channels.
    P = S['P']
    methods_sel = [m for m in _selected_methods() if ('flag_' + m in P['meta'].columns) or m == 'event']
    stages_sel  = _selected_stages()
    evt_sel     = _selected_event_types()
    dropped     = _dropped_channels()
    rule, rule_value = _epoch_rule()
    # The channel layer is engaged ONLY when the user actually uses it. Otherwise pair_flags stays None
    # and the decision is exactly tool 6's stored flags — important because the recomputed fallback
    # flags depend on the thresholds, which differ from the tool-6 run when no params JSON is present.
    use_channels = bool(dropped) or rule != 'any'
    base, meth, insc = L.recompute_reject(
        P['meta'], methods_sel, stages_sel, evt_sel,
        pair_flags=S.get('pair_flags') if use_channels else None,
        ch_names=P['ch_names'], dropped_channels=dropped,
        epoch_rule=rule, epoch_rule_value=rule_value)
    S['base_reject'] = base; S['reject_method'] = meth; S['in_scope'] = insc
    S['methods_sel'] = methods_sel; S['stages_sel'] = stages_sel; S['event_types_sel'] = evt_sel
    S['dropped_channels'] = dropped; S['epoch_rule'] = rule; S['epoch_rule_value'] = rule_value
    if S.get('pair_flags'):
        S['badness'], _ = L.channel_badness(S['pair_flags'], methods_sel, insc, len(P['ch_names']))
        _refresh_badness_label()
        _refresh_channel_labels()
    if reset_overrides or 'final_reject' not in S:
        S['final_reject'] = base.copy(); S['overridden'] = set()
    S['metrics'] = None   # selection changed -> recompute metrics on the next Build / Run

def _apply_drop_rule(*_):
    # Channel-first rule applied straight to the checkboxes: re-tick everything, then untick the
    # channels whose badness exceeds the threshold. 0 = keep every channel - the test is a strict '>',
    # so 0 would otherwise untick every channel flagged even once.
    try:
        for cb in _chan_cb.values():
            cb.value = True
        pct = float(ft_drop_pct.value)
        bad = S.get('badness')
        if pct <= 0 or bad is None:
            return
        over = set(L.channels_over_threshold(bad, S['P']['ch_names'], pct))
        for c, cb in _chan_cb.items():
            if c in over:
                cb.value = False
    except Exception as e:
        with out_load:
            print(f'Channel rule error: {e}')

ft_drop_pct.observe(_apply_drop_rule, names='value')

def _on_load(b):
    with out_load:
        clear_output(wait=True)
        try:
            fid = dd_part.value
            if not fid:
                print('Select a participant first.'); return
            p = S['parts'][fid]
            P = L.load_participant(p['fif'])
            cs = L.parse_custom_field(txt_custom.value)
            thr, info = L.load_params(p['folder'], fid, custom_stages_fallback=cs)
            S.update({'P': P, 'fid': fid, 'folder': p['folder'], 'thr0': thr, 'pinfo': info,
                      'custom_stages': cs, 'freqs': None, 'psds': None, 'metrics': None})
            S['ctx']    = L.load_context_epochs(p['folder'], fid)
            S['onsets'] = L.load_event_onsets(p['folder'], fid)   # optional montage markers (Section 3)
            S['out_folder'], S['reports_folder'] = _output_folders(p['folder'])
            # Methods present -> disable/untick the absent ones; event types -> populate sub-boxes (all ticked).
            for m in L.METHOD_ORDER:
                present = ('flag_' + m in P['meta'].columns)
                _method_cb[m].disabled = not present
                if not present:
                    _method_cb[m].value = False
            evt_types = sorted(c[len('evt_'):] for c in P['meta'].columns if c.startswith('evt_'))
            _populate_event_types(evt_types)
            _populate_channels(P['ch_names'])
            S['pair_flags'], S['flags_source'] = _load_channel_flags(fid, P, thr)
            dd_rule.value = 'any'                     # every load starts from the classic decision
            _build_amp_widgets(thr, cs)
            _set_scalar_thresholds(thr)
            # Section-2 / Section-3 defaults follow the montage DENSITY: a sparse PSG montage keeps the
            # classic behaviour (fit every epoch, draw every channel, one table row per channel), a
            # dense one starts on the high-density settings. All of them stay editable.
            hd = len(P['ch_names']) > L.HD_CHANNEL_THRESHOLD
            try:
                ft_1f_n.value = L.N_1F_SUBSAMPLE_HD if hd else 0
                dd_channels.value = 'flagged' if hd else 'all'
                ft_table_rows.value = L.DEFAULT_TABLE_ROWS if hd else 0
            except Exception:
                pass                      # Sections 2/3 not executed yet (partially-run Jupyter)
            _recompute_reject(reset_overrides=True)
            # A drop threshold left over from the previous participant must be re-applied to THIS
            # one's channels (the observer only fires on a value change), then the decision redone.
            if float(ft_drop_pct.value) > 0:
                _apply_drop_rule()
                _recompute_reject(reset_overrides=True)
            btn_apply.disabled = False
            n = len(P['stages']); nins = int(S['in_scope'].sum()); nrej = int(S['base_reject'].sum())
            src = 'from params JSON' if info['found'] else 'DEFAULTS (no *_preprocessing_params.json found)'
            n_ch = len(P['ch_names'])
            print(f'Loaded {fid}:  {n} epochs, {nins} in-scope, {nrej} rejected by selection '
                  f'({100*nrej/max(nins,1):.1f}% of in-scope)')
            print(f'  channels = {P["ch_names"]}   |   sfreq = {P["sfreq"]:.0f} Hz   |   '
                  f'epoch length = {info["epoch_length_s"]} s')
            print(f'  methods present = {P["methods_present"]}   |   selected = {S["methods_sel"]}')
            print(f'  stages selected = {S["stages_sel"]}')
            if evt_types:
                print(f'  event types = {evt_types}  (selected = {S["event_types_sel"]})')
            if S['ctx'] is not None:
                print(f'  context channels = {S["ctx"]["labels"]}  (toggle in Section 3)')
            else:
                print('  context channels = none (no *_context-epo.fif companion from tool 6)')
            if S['onsets'] is not None:
                print(f'  event onsets = {len(S["onsets"])} marks (shown on the Section-3 montage)')
            print(f'  thresholds = {src}')
            print(f'  per-channel flags = {S["flags_source"]}')
            if cs:
                print(f'  custom stages = {cs}')
            # High-density montages: warn about the memory footprint and about the 'any channel' rule,
            # whose rejection rate explodes with the channel count (see the Channels box above).
            if n_ch > L.HD_CHANNEL_THRESHOLD:
                gb = P['data_uV'].nbytes / 1e9
                print(f'  NOTE: high-density montage ({n_ch} channels) — ~{2*gb:.1f} GB resident '
                      f'(signal copy {gb:.1f} GB + the epochs object). Re-run tool 6 with resampling '
                      f'to reduce it.')
                bad = S.get('badness')
                if bad is not None and (bad > DROP_PCT_SUGGESTED).any():
                    over = L.channels_over_threshold(bad, P['ch_names'], DROP_PCT_SUGGESTED)
                    print(f'  NOTE: {len(over)} channel(s) flagged on more than {DROP_PCT_SUGGESTED:g}% '
                          f'of the in-scope epochs: {over}. Every channel is KEPT by default - set '
                          f'"Drop channels flagged in more than (%) of the epochs" in Section 1 to drop '
                          f'them, and/or switch the epoch rule away from "any flagged channel", then '
                          f'click "Apply selection".')
            if _is_processed(fid, p['folder']):
                print('  NOTE: a manual clean-epo + review already exist on disk (they will be overwritten on save).')
            print('\nProceed to Section 2 (report) or Section 3 (navigator). '
                  'Change a tick then click "Apply selection" to redecide.')
        except Exception as e:
            print(f'Load error: {e}')

def _on_apply(b):
    with out_load:
        try:
            if 'P' not in S:
                print('Load a participant first.'); return
            _recompute_reject(reset_overrides=True)
            nins = int(S['in_scope'].sum()); nrej = int(S['base_reject'].sum())
            dropped = S['dropped_channels']; rule = S['epoch_rule']
            n_kept = len(S['P']['ch_names']) - len(dropped)
            rule_txt = ('any flagged channel' if rule == 'any'
                        else f'> {S["epoch_rule_value"]:g}% of the kept channels')
            print(f'Applied selection: {nins} in-scope, {nrej} rejected ({100*nrej/max(nins,1):.1f}%). '
                  f'methods={S["methods_sel"]}  stages={S["stages_sel"]}  events={S["event_types_sel"]}. '
                  f'channels kept={n_kept}/{len(S["P"]["ch_names"])}'
                  + (f' (dropped: {dropped})' if dropped else '')
                  + f'  epoch rule={rule_txt}. '
                  f'Manual overrides reset -> re-run Section 2 / Section 3.')
            if n_kept == 0:
                print('  ⚠ Every channel is unticked — nothing can be flagged and the clean-epo would be '
                      'empty. Tick at least one channel.')
        except Exception as e:
            print(f'Apply error: {e}')

btn_load.on_click(_on_load)
btn_apply.on_click(_on_apply)

display(widgets.VBox([
    fc_data, fc_raw, fc_reports,
    widgets.HBox([dd_part, txt_custom]), lbl_done,
    box_stages, box_methods, box_evt,
    box_triage,
    widgets.HBox([btn_load, btn_apply]),
    lbl_scan, out_load,
]))


## Section 2 — Global per-stage report

Per sleep stage: **PSD overlay** (rejected epochs coloured by method over clean epochs + IQR band),
**metric distributions** (clean vs rejected with threshold lines) and a **stage × method rejection
table**, plus a **channels × epochs flagging heatmap with a per-channel badness bar** (which electrode
carries the night's flags). The thresholds below are the ones **tool 6 actually used** (from the
participant's params, or tool-6 defaults) and are **read-only** unless you tick *Override thresholds*:
they drive the reference lines and the per-channel attribution, never the flags stored in the `.fif`.
*1/f fitting runs per (epoch × channel): ~1 min on a 3-channel night, ~8 min on a 32-channel one —
hence the subsample and parallel options, both stated in the figure when they change what is shown.*

In [ ]:
# Threshold widgets (reference lines; editable). Amplitude widgets are rebuilt per participant.
box_amp   = widgets.HBox([])
_amp_w    = {}
ft_flat   = widgets.FloatText(description='Flat <', value=1.0,   style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_grad   = widgets.FloatText(description='Grad >', value=100.0, style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_mae    = widgets.FloatText(description='MAE >',  value=0.15,  style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_r2     = widgets.FloatText(description='R² <',   value=0.95,  style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))

# --- 1/f fitting cost (matters on a HIGH-DENSITY montage) ------------------------------------------
# The fit runs per (in-scope epoch x channel): ~1 min on a 3-channel night, ~8 min on a 32-channel one.
# Two levers, both reported in the figure so a reader never mistakes a subsample for the full night:
#   * Subsample  -> fit a seeded RANDOM sample of in-scope epochs. Only the metric DISTRIBUTIONS use
#                   these values; the keep/reject flags always come from the .fif, so a subsample can
#                   never change a decision. Default N is set at load: 250 above the channel threshold,
#                   0 (= every epoch) below it, so a sparse PSG montage is unaffected.
#   * Parallel   -> joblib over epochs. EXACT (same numbers), just faster; falls back to serial on error.
cb_1f_sub = widgets.Checkbox(value=True, description='Subsample 1/f epochs', indent=False,
                             style={'description_width':'initial'}, layout=widgets.Layout(width='190px'))
ft_1f_n   = widgets.IntText(description='max', value=0, style={'description_width':'35px'},
                            layout=widgets.Layout(width='120px'))
# Revealed by the checkbox: the initial display MUST be derived from its value, never hard-coded to
# 'none' — an observer only fires on a CHANGE, and this checkbox starts TICKED (SPEC: 'Checkbox-revealed
# parameter widgets'; this is exactly the bug that hid the Curry tool-5 high-pass box).
ft_1f_n.layout.display = '' if cb_1f_sub.value else 'none'
def _toggle_1f_sub(change):
    ft_1f_n.layout.display = '' if change['new'] else 'none'
cb_1f_sub.observe(_toggle_1f_sub, names='value')
cb_1f_par = widgets.Checkbox(value=True, description='Parallel fit', indent=False,
                             style={'description_width':'initial'}, layout=widgets.Layout(width='140px'))
box_1f = widgets.HBox([widgets.HTML('<b>1/f fit:</b>'), cb_1f_sub, ft_1f_n, cb_1f_par])

def _build_amp_widgets(thr, custom_stages):
    global _amp_w
    _amp_w = {}
    order = ['W', 'N1', 'N2', 'N3', 'R'] + [c for c in custom_stages if c not in ('W','N1','N2','N3','R')]
    items = []
    for st in order:
        v = float(thr['amplitude_ptp_uV'].get(st, 250.0))
        w = widgets.FloatText(description=st, value=v, style={'description_width':'40px'},
                              layout=widgets.Layout(width='120px'))
        w.observe(_refresh_thr_status, names='value')     # keep the drift warning in sync
        _amp_w[st] = w
        items.append(w)
    box_amp.children = [widgets.HTML('<b>Amplitude p-p (µV) per stage:</b>')] + items
    _apply_thr_lock()            # rebuilt per participant -> re-apply the read-only lock

def _set_scalar_thresholds(thr):
    ft_flat.value = float(thr['flat_ptp_uV'])
    ft_grad.value = float(thr['gradient_uV_per_sample'])
    ft_mae.value  = float(thr['1f_mae_max'])
    ft_r2.value   = float(thr['1f_r2_min'])

def _collect_thresholds():
    return {
        'amplitude_ptp_uV': {st: float(w.value) for st, w in _amp_w.items()},
        'flat_ptp_uV': float(ft_flat.value),
        'gradient_uV_per_sample': float(ft_grad.value),
        '1f_mae_max': float(ft_mae.value),
        '1f_r2_min': float(ft_r2.value),
    }

# --- Threshold edition (these are TOOL 6's thresholds, not tool 7's) -------------------------------
# The values above are the ones tool 6 actually used (params JSON, or tool-6 defaults when the sidecar
# is missing). They drive the reference lines, the per-channel attribution shown in Section 3 and -
# when no reports folder is selected - the per-(epoch, channel) flags recomputed from the signal, which
# do feed the decision as soon as the channel layer is engaged. Editing them must therefore be a
# deliberate act: read-only until 'Override thresholds' is ticked, with a warning as soon as a value
# drifts away from the tool-6 run.
cb_thr_override = widgets.Checkbox(value=False, description='Override thresholds (advanced)',
                                   indent=False, style={'description_width':'initial'},
                                   layout=widgets.Layout(width='270px'))
btn_thr_reset = widgets.Button(description='Reset to tool-6 values', icon='undo', disabled=True,
                               layout=widgets.Layout(width='200px'))
lbl_thr_src  = widgets.HTML('')
lbl_thr_warn = widgets.HTML('')

def _thr_widgets():
    return [ft_flat, ft_grad, ft_mae, ft_r2] + list(_amp_w.values())

def _refresh_thr_status(*_):
    # Provenance line + an amber warning as soon as a value differs from the loaded (tool-6) one.
    info = S.get('pinfo')
    if info is None:
        lbl_thr_src.value = '<small>Thresholds: load a participant first.</small>'
        lbl_thr_warn.value = ''
        return
    src = ('read from <code>*_preprocessing_params.json</code>' if info.get('found')
           else 'tool-6 DEFAULTS (no <code>*_preprocessing_params.json</code> found)')
    lbl_thr_src.value = (f'<small>Thresholds: {src} &mdash; they drive the reference lines and the '
                         f'per-channel attribution, not the stored flags.</small>')
    thr0 = S.get('thr0')
    drift = False
    try:
        if thr0 is not None:
            cur = _collect_thresholds()
            drift = (any(abs(cur[k] - float(thr0[k])) > 1e-9
                         for k in ('flat_ptp_uV', 'gradient_uV_per_sample', '1f_mae_max', '1f_r2_min'))
                     or any(abs(v - float(thr0['amplitude_ptp_uV'].get(st, 250.0))) > 1e-9
                            for st, v in cur['amplitude_ptp_uV'].items()))
    except Exception:
        drift = False
    lbl_thr_warn.value = ('<span style="color:#e65100;">&#9888; Thresholds differ from the tool-6 run '
                          '&mdash; the reference lines and the per-channel attribution no longer match '
                          'the flags stored in the .fif.</span>' if drift else '')

def _apply_thr_lock(*_):
    # Read-only unless the user ticked the override box (the values stay visible and stay in use).
    for w in _thr_widgets():
        w.disabled = not cb_thr_override.value
    btn_thr_reset.disabled = not cb_thr_override.value
    _refresh_thr_status()

def _on_thr_reset(b):
    try:
        thr0 = S.get('thr0')
        if thr0 is None:
            return
        _set_scalar_thresholds(thr0)
        for st, w in _amp_w.items():
            w.value = float(thr0['amplitude_ptp_uV'].get(st, 250.0))
        _refresh_thr_status()
    except Exception as e:
        with out_report:
            print(f'Threshold reset error: {e}')

cb_thr_override.observe(_apply_thr_lock, names='value')
btn_thr_reset.on_click(_on_thr_reset)
for _w in (ft_flat, ft_grad, ft_mae, ft_r2):
    _w.observe(_refresh_thr_status, names='value')
_apply_thr_lock()

prog_report = widgets.IntProgress(description='1/f fit:', min=0, max=1, value=0,
                                  layout=widgets.Layout(width='320px'))
btn_run_report  = widgets.Button(description='Build, show & save HTML', button_style='primary', icon='chart-area')
btn_save_report = widgets.Button(description='Save HTML only (no inline)', icon='save',
                                 layout=widgets.Layout(width='230px'))

def _ensure_metrics(thr, verbose=True):
    # Welch PSDs + per-epoch metric values, computed once per selection (shared by both buttons).
    P = S['P']
    fit_fmin, fit_fmax = S['pinfo']['fit_range']
    if S.get('freqs') is None:
        if verbose:
            print('Computing Welch PSDs…')
        S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=fit_fmax,
                                               smoothing=S['pinfo'].get('psd_smoothing'))
    if S.get('metrics') is None:
        in_scope = S['in_scope']
        do_1f = ('1f_r2' in S['methods_sel']) or ('1f_error' in S['methods_sel'])
        sub_n = int(ft_1f_n.value) if cb_1f_sub.value else 0
        n_jobs = -1 if cb_1f_par.value else 1
        n_fit = int(in_scope.sum())
        if sub_n and n_fit > sub_n:
            n_fit = sub_n
        if verbose:
            print(f'Fitting 1/f on {n_fit} in-scope epoch(s)…' if do_1f
                  else '1/f methods not selected — skipping the 1/f fit.')
        prog_report.max = max(1, n_fit); prog_report.value = 0
        S['metrics'] = L.compute_epoch_metrics(P['data_uV'], S['freqs'], S['psds'], fmin=fit_fmin,
                                               fit_mask=in_scope, do_1f=do_1f,
                                               subsample_n=sub_n, n_jobs=n_jobs,
                                               progress=lambda d, t: setattr(prog_report, 'value', d))
    return S['metrics']

def _build_figs(thr):
    # Section-2 figures + table for the CURRENT selection, including the channels x epochs flagging
    # heatmap (per-channel overview — the decisive view on a dense montage).
    P = S['P']
    stage_order = L.ordered_present_stages(P['stages'][S['in_scope']], S['custom_stages'])
    return L.build_participant_report_figs(
        P, S['metrics'], S['freqs'], S['psds'], thr, S['custom_stages'],
        reject_flag=S['base_reject'], reject_method=S['reject_method'],
        stage_order=stage_order, methods=S['methods_sel'],
        pair_flags=S.get('pair_flags'), in_scope=S['in_scope'], badness_pct=S.get('badness'),
        dropped_channels=S.get('dropped_channels', ()), flags_source=S.get('flags_source', ''))

def _on_run_report(b):
    with out_report:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            thr = _collect_thresholds()
            _ensure_metrics(thr)
            figs, html = _build_figs(thr)
            clear_output(wait=True)
            display(HTML(html))
            for name, fig in figs:
                show_fig(fig, close=False)          # keep open so we can also save them below
            # Build == save: write the HTML report right away (no second click needed).
            try:
                S['reports_folder'].mkdir(parents=True, exist_ok=True)
                out = S['reports_folder'] / f'{S["fid"]}_qc2b_report.html'
                L.save_report_html(out, f'{S["fid"]} — QC of rejected epochs', figs, html)
                print(f'Saved report → {out}')
            except Exception as e:
                print(f'⚠ Report shown but HTML save failed: {e}')
            btn_save_report.disabled = False
        except Exception as e:
            print(f'Report error: {e}')

def _on_save_report(b):
    # Standalone: computes PSDs / 1-f metrics if Build was not clicked, then writes the HTML
    # without rendering anything inline (keeps the notebook light).
    with out_report:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            thr = _collect_thresholds()
            _ensure_metrics(thr)
            figs, html = _build_figs(thr)
            S['reports_folder'].mkdir(parents=True, exist_ok=True)
            out = S['reports_folder'] / f'{S["fid"]}_qc2b_report.html'
            L.save_report_html(out, f'{S["fid"]} — QC of rejected epochs', figs, html)
            print(f'Saved report → {out}')
        except Exception as e:
            print(f'Save error: {e}')

btn_run_report.on_click(_on_run_report)
btn_save_report.on_click(_on_save_report)

display(widgets.VBox([
    widgets.HBox([cb_thr_override, btn_thr_reset]),
    lbl_thr_src,
    box_amp,
    widgets.HBox([ft_flat, ft_grad, ft_mae, ft_r2]),
    lbl_thr_warn,
    box_1f,
    widgets.HBox([btn_run_report, prog_report, btn_save_report]),
    out_report,
]))


## Section 3 — Per-epoch navigator

Walk through the rejected epochs one by one. **Montage** (± context) with the flagging cause highlighted
per channel, plus a **detail panel** (PSD + aperiodic fit, band power + 50 Hz ratio, epoch spectrogram,
per-channel metric table). Use the **keep / reject** toggle to override the decision — overrides feed
Section 4.

On a dense montage, **Channels** restricts the montage to the flagged electrodes (the title always
states how many of how many are drawn), **Table rows** keeps the metric table readable, and the detail
panel gains **topomaps** of peak-to-peak and 1/f MAE when the recording carries electrode positions
(Curry does; Compumedics EDF does not) — a single bad electrode reads as a hot spot, a whole-head
artefact as a global shift.

In [ ]:
dd_stage   = widgets.Dropdown(description='Stage:', options=['(all)'], value='(all)',
                              style={'description_width':'55px'}, layout=widgets.Layout(width='170px'))
dd_method  = widgets.Dropdown(description='Method:', options=['(all)'], value='(all)',
                              style={'description_width':'60px'}, layout=widgets.Layout(width='190px'))
dd_spectro = widgets.Dropdown(description='Spectro ch:', options=[], style={'description_width':'80px'},
                              layout=widgets.Layout(width='230px'))
# --- High-density display levers -------------------------------------------------------------------
# On a 32-64 channel montage the median rejected epoch is flagged on ONE channel, so drawing all of
# them to find it is backwards; and a 32-row metric table is illegible. Both default to the classic
# 'show everything' and are switched at load when the montage turns out to be dense.
dd_channels = widgets.Dropdown(description='Channels:', value='all',
                               options=[('all', 'all'),
                                        ('flagged only', 'flagged'),
                                        ('flagged + neighbours', 'flagged+neighbours'),
                                        ('worst 16 (p-p)', 'worst')],
                               style={'description_width':'70px'}, layout=widgets.Layout(width='250px'))
ft_table_rows = widgets.IntText(description='Table rows', value=0, style={'description_width':'80px'},
                                layout=widgets.Layout(width='160px'))
sl_ctx     = widgets.IntSlider(description='± context', value=1, min=0, max=3, layout=widgets.Layout(width='230px'))
cb_context = widgets.Checkbox(value=True, description='Show EOG/EMG context', indent=False,
                              layout=widgets.Layout(width='210px'))
sl_epoch   = widgets.IntSlider(description='Index', value=0, min=0, max=0, continuous_update=False,
                               layout=widgets.Layout(width='400px'))
btn_prev   = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next   = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
tgl_keep   = widgets.ToggleButtons(options=[('Keep','keep'), ('Reject','reject')], value='reject',
                                   style={'button_width':'90px'})
btn_apply_all = widgets.Button(description='Apply to all shown', icon='clone',
                               tooltip='Set the current Keep/Reject decision on every epoch in the current '
                                       'stage + method filter', layout=widgets.Layout(width='190px'))
lbl_applyall = widgets.HTML('')
lbl_pos    = widgets.HTML('')
btn_run_nav = widgets.Button(description='Start navigator', button_style='primary', icon='play')
out_detail = widgets.Output()     # PSD + metric table + band power + spectrogram (2x2 figure)

# Fixed display scales (µV peak-to-peak per channel row), clinical PSG conventions from the shared lib.
# They do NOT follow the epoch: keeping them constant is what makes waveform amplitudes (75 µV slow waves,
# EMG tone, ...) comparable from one epoch to the next. Editable here if a montage needs more/less room.
_sclay = widgets.Layout(width='130px')
_scsty = {'description_width': '46px'}
ft_sc_eeg = widgets.FloatText(description='EEG', value=L.DISPLAY_SCALE_UV['EEG'], step=25,
                              style=_scsty, layout=_sclay)
ft_sc_eog = widgets.FloatText(description='EOG', value=L.DISPLAY_SCALE_UV['EOG'], step=50,
                              style=_scsty, layout=_sclay)
ft_sc_emg = widgets.FloatText(description='EMG', value=L.DISPLAY_SCALE_UV['EMG'], step=25,
                              style=_scsty, layout=_sclay)
ft_sc_ecg = widgets.FloatText(description='ECG', value=L.DISPLAY_SCALE_UV['ECG'], step=100,
                              style=_scsty, layout=_sclay)
box_scales = widgets.HBox([widgets.HTML('<b>Scale (µV/row):</b>'),
                           ft_sc_eeg, ft_sc_eog, ft_sc_emg, ft_sc_ecg])

def _current_scales():
    return {'EEG': ft_sc_eeg.value, 'EOG': ft_sc_eog.value,
            'EMG': ft_sc_emg.value, 'ECG': ft_sc_ecg.value}

def _method_hit_array(m):
    # Boolean (n_ep,) mask: epochs that method m flags (for 'event', OR of the selected evt_<type>).
    P = S['P']; n = len(P['stages'])
    if m == 'event':
        ets = S.get('event_types_sel') or []
        cols = ['evt_' + t for t in ets if 'evt_' + t in P['meta'].columns]
        if cols:
            arr = np.zeros(n, dtype=bool)
            for c in cols:
                arr |= P['meta'][c].astype(bool).values
            return arr
        return (P['meta']['flag_event'].astype(bool).values if 'flag_event' in P['meta'].columns
                else np.zeros(n, dtype=bool))
    return (P['meta']['flag_' + m].astype(bool).values if ('flag_' + m in P['meta'].columns)
            else np.zeros(n, dtype=bool))

def _build_method_arrays():
    S['method_arrays'] = {m: _method_hit_array(m) for m in S['methods_sel']}
    # Per-event-type masks so the Method dropdown can offer 'event: <type>' and Apply-to-all can act on a
    # single event category (e.g. Keep every epoch flagged by hypopneas once they look clean on the EEG).
    P = S['P']
    S['event_type_arrays'] = {t: P['meta']['evt_' + t].astype(bool).values
                              for t in (S.get('event_types_sel') or [])
                              if 'evt_' + t in P['meta'].columns}

def _nav_list():
    P = S['P']; rej = list(np.where(S['base_reject'])[0])
    st = dd_stage.value; m = dd_method.value
    if st != '(all)':
        rej = [ei for ei in rej if P['stages'][ei] == st]
    if m != '(all)':
        if m == 'event (any)':
            arr = S.get('method_arrays', {}).get('event')
        elif m.startswith('event: '):
            arr = S.get('event_type_arrays', {}).get(m[len('event: '):])
        else:
            arr = S.get('method_arrays', {}).get(m)
        if arr is not None:
            rej = [ei for ei in rej if arr[ei]]
    return rej

def _refresh_filter_options():
    P = S['P']; base = S['base_reject']
    stages = L.ordered_present_stages(P['stages'][base], S['custom_stages'])
    # Non-event methods first, then a generic 'event (any)' and one 'event: <type>' per event category
    # that actually flagged an in-scope epoch (so a specific event type can be reviewed / bulk-kept).
    non_evt = [m for m in S['methods_sel'] if m != 'event' and (S['method_arrays'][m] & base).any()]
    evt_opts = []
    ev_arr = S['method_arrays'].get('event')
    if ev_arr is not None and (ev_arr & base).any():
        evt_opts.append('event (any)')
        evt_opts += ['event: ' + t for t in (S.get('event_types_sel') or [])
                     if t in S.get('event_type_arrays', {}) and (S['event_type_arrays'][t] & base).any()]
    dd_stage.unobserve(_on_filter, 'value'); dd_method.unobserve(_on_filter, 'value')
    dd_stage.options = ['(all)'] + stages; dd_stage.value = '(all)'
    dd_method.options = ['(all)'] + non_evt + evt_opts; dd_method.value = '(all)'
    dd_stage.observe(_on_filter, 'value'); dd_method.observe(_on_filter, 'value')

def _current_ei():
    nav = S.get('nav_list', [])
    if not nav:
        return None
    return nav[min(sl_epoch.value, len(nav) - 1)]

def _methods_of(ei):
    # Names of the SELECTED methods that flagged epoch ei (resolves 'multiple').
    return [m for m in S['methods_sel'] if S.get('method_arrays', {}).get(m) is not None
            and S['method_arrays'][m][ei]]

def _montage_channels(ei, thr):
    # Channels the montage draws: the dropdown mode, minus the channels unticked in Section 1 (a
    # dropped channel is out of the decision, so it must not appear as flagging the epoch).
    # None -> every channel (kept explicit so the classic path stays byte-identical).
    P = S['P']
    dropped = set(S.get('dropped_channels', ()))
    if dd_channels.value == 'all' and not dropped:
        return None
    sel = L.select_montage_channels(P, ei, thr, mode=dd_channels.value)
    if dropped:
        sel = [i for i in sel if P['ch_names'][i] not in dropped]
    return sel or None

def _render_montage(ei, thr):
    with out_nav:
        clear_output(wait=True)
        try:
            ctx = S.get('ctx') if cb_context.value else None
            f1 = L.plot_epoch_montage(S['P'], ei, thr, context=int(sl_ctx.value), ctx=ctx,
                                      onsets=S.get('onsets'), title_method=S['reject_method'][ei],
                                      scales=_current_scales(),
                                      show_channels=_montage_channels(ei, thr))
            show_fig(f1)
        except Exception as e:
            print(f'Montage error: {e}')

def _render_detail(ei=None, thr=None):
    if ei is None:
        ei = _current_ei()
    if ei is None:
        return
    if thr is None:
        thr = _collect_thresholds()
    with out_detail:
        clear_output(wait=True)
        try:
            P = S['P']
            spi = P['ch_names'].index(dd_spectro.value) if dd_spectro.value in P['ch_names'] else 0
            # clean reference for the PSD background = in-scope epochs NOT rejected by the selection
            clean_ref = np.asarray(S['in_scope'], dtype=bool) & ~np.asarray(S['base_reject'], dtype=bool)
            f2 = L.plot_epoch_detail(P, ei, S['freqs'], S['psds'], thr, spectro_ch_idx=spi,
                                     fmin=S['pinfo']['fit_range'][0], clean_mask=clean_ref,
                                     table_rows=int(ft_table_rows.value) or None)
            show_fig(f2)
        except Exception as e:
            print(f'Detail error: {e}')

def _render():
    nav = S.get('nav_list', [])
    if not nav:
        with out_nav:
            clear_output(wait=True); print('No rejected epochs for this filter.')
        with out_detail:
            clear_output(wait=True)
        lbl_pos.value = ''; lbl_applyall.value = ''
        return
    ei = _current_ei()
    thr = _collect_thresholds()
    P = S['P']
    fr = 'reject' if S['final_reject'][ei] else 'keep'
    tgl_keep.unobserve(_on_toggle, 'value'); tgl_keep.value = fr; tgl_keep.observe(_on_toggle, 'value')
    ov = ' (overridden)' if ei in S['overridden'] else ''
    # Which event type(s) flagged this epoch (tool-6 evt_<type> metadata columns).
    evt_here = [c[len('evt_'):] for c in P['meta'].columns
                if c.startswith('evt_') and bool(P['meta'].loc[ei, c])]
    evt_txt = f' — events: <b>{", ".join(evt_here)}</b>' if evt_here else ''
    # reject_method, resolving 'multiple' to the actual selected methods that flagged it.
    rm = S['reject_method'][ei]
    if rm == 'multiple':
        rm = f'multiple ({", ".join(_methods_of(ei))})'
    lbl_pos.value = (f'<b>Epoch {ei}</b> — {sl_epoch.value+1}/{len(nav)} — stage {P["stages"][ei]} '
                     f'— reject_method={rm}{evt_txt} — decision: <b>{fr}</b>{ov}')
    # Homogeneity hint for 'Apply to all shown'.
    n_rej = int(sum(1 for e in nav if S['final_reject'][e])); n_keep = len(nav) - n_rej
    homog = ('all <b>reject</b>' if n_keep == 0 else 'all <b>keep</b>' if n_rej == 0
             else f'MIXED — {n_keep} keep / {n_rej} reject')
    lbl_applyall.value = (f'<small>Shown set: {len(nav)} epochs — {homog}. '
                          f'&ldquo;Apply to all shown&rdquo; sets the current toggle on all of them.</small>')
    _render_montage(ei, thr)
    _render_detail(ei, thr)

def _on_run_nav(b):
    try:
        if 'P' not in S:
            with out_nav: clear_output(); print('Load a participant first (Section 1).'); return
        P = S['P']
        if S['freqs'] is None:
            S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=S['pinfo']['fit_range'][1],
                                                   smoothing=S['pinfo'].get('psd_smoothing'))
        dd_spectro.options = P['ch_names']; dd_spectro.value = P['ch_names'][0]
        _build_method_arrays()
        _refresh_filter_options()
        S['nav_list'] = _nav_list()
        sl_epoch.max = max(0, len(S['nav_list']) - 1); sl_epoch.value = 0
        _render()
    except Exception as e:
        with out_nav: clear_output(); print(f'Navigator error: {e}')

def _on_filter(chg):
    try:
        S['nav_list'] = _nav_list()
        sl_epoch.max = max(0, len(S['nav_list']) - 1); sl_epoch.value = 0
        _render()
    except Exception as e:
        with out_nav: print(f'Filter error: {e}')

def _step(delta):
    def _f(b):
        sl_epoch.value = int(np.clip(sl_epoch.value + delta, 0, sl_epoch.max))
    return _f

def _set_decision(ei, reject):
    S['final_reject'][ei] = bool(reject)
    if S['final_reject'][ei] != bool(S['base_reject'][ei]):
        S['overridden'].add(ei)
    else:
        S['overridden'].discard(ei)

def _on_toggle(chg):
    try:
        ei = _current_ei()
        if ei is None:
            return
        _set_decision(ei, chg['new'] == 'reject')
        _render()
    except Exception as e:
        with out_nav: print(f'Toggle error: {e}')

def _on_apply_all(b):
    # Apply the current Keep/Reject toggle to EVERY epoch in the current stage + method filter.
    try:
        nav = S.get('nav_list', [])
        if not nav:
            return
        reject = (tgl_keep.value == 'reject')
        for ei in nav:
            _set_decision(ei, reject)
        _render()
    except Exception as e:
        with out_nav: print(f'Apply-all error: {e}')

btn_run_nav.on_click(_on_run_nav)
dd_stage.observe(_on_filter, 'value')
dd_method.observe(_on_filter, 'value')
sl_epoch.observe(lambda chg: _render(), 'value')
sl_ctx.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                            if _current_ei() is not None else None), 'value')
cb_context.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                                if _current_ei() is not None else None), 'value')
dd_spectro.observe(lambda chg: _render_detail(), 'value')
dd_channels.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                                 if _current_ei() is not None else None), 'value')
ft_table_rows.observe(lambda chg: _render_detail(), 'value')
for _ft in (ft_sc_eeg, ft_sc_eog, ft_sc_emg, ft_sc_ecg):
    _ft.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                             if _current_ei() is not None else None), 'value')
btn_prev.on_click(_step(-1)); btn_next.on_click(_step(+1))
btn_apply_all.on_click(_on_apply_all)
tgl_keep.observe(_on_toggle, 'value')

display(widgets.VBox([
    widgets.HBox([btn_run_nav, dd_stage, dd_method, sl_ctx, cb_context]),
    widgets.HBox([btn_prev, sl_epoch, btn_next]),
    widgets.HBox([dd_channels, ft_table_rows]),
    box_scales,
    widgets.HBox([widgets.HTML('<b>Decision:</b>'), tgl_keep, btn_apply_all]),
    lbl_applyall, lbl_pos,
    out_nav,
    out_detail,
    # Indented to sit under the spectrogram: the detail figure renders ~1105 px wide at dpi=100 and its
    # bottom-right (spectrogram) panel starts ~695 px in, so the selector lines up with the plot it drives.
    widgets.HBox([widgets.HTML('<b>Spectrogram channel:</b>'), dd_spectro],
                 layout=widgets.Layout(margin='0 0 0 695px')),
]))



## Section 4 — Manual override & save

Review the final keep/reject decision (with your overrides) and export the validated epochs.

**Data** → `derivatives/clean_epo_manual/<subtree>/{file_id}_clean-epo.fif` — the **in-scope kept** epochs
only (i.e. the selected stages, minus the rejected ones), with any **channel unticked in Section 1
removed** from the file.

**Reports** → `reports_rejection_manual/<subtree>/` (beside `reports_preprocessing/`):
`{file_id}_epoch_rejection_reviewed.tsv` (per-epoch metadata + decision columns),
`{file_id}_qc2b_review_log.tsv` (one row per overridden epoch),
`{file_id}_manualreject_decision.tsv` (one-row durable record: counts, %, overrides, stages/methods/event
types used, thresholds) and `{file_id}_manualreject_report.html` (visual recap of the rejected epochs).
A `global_manualreject_summary.tsv` at the root of `reports_rejection_manual/` is **rebuilt from every
decision TSV found on disk** at each save. **Tool-6 outputs are never modified.**

In [ ]:
btn_review = widgets.Button(description='Refresh review', icon='eye')
btn_save   = widgets.Button(description='Save clean-epo + report', button_style='success', icon='save')

def _rebuild_global_summary(reports_root):
    """Global summary rebuilt by GLOBBING the per-file decision TSVs on disk (interruption-safe, same
    convention as tool 7bis) — never from an in-memory list."""
    rows = []
    for f in sorted(Path(reports_root).rglob('*_manualreject_decision.tsv')):
        try:
            rows.append(pd.read_csv(f, sep='\t'))
        except Exception:
            pass
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def _on_review(b):
    with out_review:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P = S['P']; fr = np.asarray(S['final_reject'], dtype=bool)
            insc = np.asarray(S['in_scope'], dtype=bool); base = np.asarray(S['base_reject'], dtype=bool)
            n = len(fr); n_excl = int((~insc).sum())
            keep = int((insc & ~fr).sum()); rej = int((insc & fr).sum())
            rescued = int(np.sum(insc & base & ~fr)); added = int(np.sum(insc & ~base & fr))
            print(f'Epochs: {n} | in-scope: {int(insc.sum())} | excluded (out-of-scope stages): {n_excl}')
            print(f'  final keep: {keep} | final reject: {rej} | rescued: {rescued} | newly rejected: {added} '
                  f'| overrides: {len(S["overridden"])}')
            if S.get('dropped_channels'):
                print(f'  channels dropped from the clean-epo: {S["dropped_channels"]} '
                      f'({len(P["ch_names"]) - len(S["dropped_channels"])} kept)')
            if S.get('epoch_rule', 'any') != 'any':
                print(f'  epoch rule: {S["epoch_rule"]} {S.get("epoch_rule_value")}')
            fig = L.plot_review_strip(P, fr, S['overridden'], custom_stages=S['custom_stages'], in_scope=S['in_scope'])
            show_fig(fig)
        except Exception as e:
            print(f'Review error: {e}')

def _on_save(b):
    with out_review:
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P = S['P']; fr = np.asarray(S['final_reject'], dtype=bool)
            fid = S['fid']
            # Guard: unticking every channel would write a 0-channel file (MNE raises). It happens
            # easily on a SPARSE montage, where a 20% badness threshold can exceed every channel —
            # the same documented edge case as tool 7bis's channel-first step.
            if len(S.get('dropped_channels', ())) >= len(P['ch_names']):
                print('⚠ Every channel is unticked — nothing to save. Tick at least one channel in '
                      'Section 1 (or raise the channel-drop threshold; 0 = keep every channel), '
                      'then click "Apply selection".')
                return
            folder = S['out_folder']; reports_folder = S['reports_folder']
            reports_root = Path(S['data_root']) / 'reports_rejection_manual'
            folder.mkdir(parents=True, exist_ok=True)
            reports_folder.mkdir(parents=True, exist_ok=True)
            insc = np.asarray(S['in_scope'], dtype=bool)
            base = np.asarray(S['base_reject'], dtype=bool)
            thr = _collect_thresholds()

            # --- DATA first (durable per-item outputs), report afterwards ---
            meta = P['meta'].copy()
            meta['in_scope'] = insc
            meta['base_reject'] = base
            meta['reject_method_sel'] = S['reject_method']
            meta['manual_override'] = [i in S['overridden'] for i in range(len(fr))]
            meta['final_reject'] = fr
            dropped = [c for c in S.get('dropped_channels', ()) ]
            # Channel-triage provenance, carried in the metadata like tool 7bis's auto_reject_* columns
            # (constant per file; empty / 'any' on the classic path).
            meta['manual_dropped_channels'] = '+'.join(dropped)
            meta['manual_epoch_rule'] = S.get('epoch_rule', 'any')
            epochs = P['epochs'].copy(); epochs.metadata = meta
            keep_idx = np.where(insc & ~fr)[0]   # clean-epo = in-scope kept epochs only (selected stages)
            clean = epochs[keep_idx]
            if dropped:
                # Channels unticked in Section 1 are excluded from the decision, so they are also
                # removed from the clean-epo — same semantics as tool 7bis's channel-first drop.
                clean.drop_channels([c for c in dropped if c in clean.ch_names])
            clean.save(str(folder / f'{fid}_clean-epo.fif'), overwrite=True, verbose=False)
            meta.to_csv(reports_folder / f'{fid}_epoch_rejection_reviewed.tsv', sep='\t', index=False)
            rows = []
            for i in sorted(S['overridden']):
                action = 'rescued' if (base[i] and not fr[i]) else \
                         ('added' if (not base[i] and fr[i]) else 'unchanged')
                rows.append({'epoch_idx': int(i), 'stage': P['stages'][i],
                             'orig_reject': bool(base[i]), 'final_reject': bool(fr[i]),
                             'action': action})
            pd.DataFrame(rows, columns=['epoch_idx','stage','orig_reject','final_reject','action']) \
                .to_csv(reports_folder / f'{fid}_qc2b_review_log.tsv', sep='\t', index=False)
            # one-row durable decision record (source of the global summary), written BEFORE the report
            decision = L.build_manual_decision_row(
                fid, P, base, fr, insc, S['overridden'], S['stages_sel'], S['methods_sel'],
                S['event_types_sel'], thr, epoch_length_s=S['pinfo']['epoch_length_s'],
                dropped_channels=dropped, epoch_rule=S.get('epoch_rule', 'any'),
                epoch_rule_value=S.get('epoch_rule_value', 20.0),
                flags_source=S.get('flags_source', ''))
            decision.to_csv(reports_folder / f'{fid}_manualreject_decision.tsv', sep='\t', index=False)
            print(f'Saved {len(keep_idx)} clean epochs × {len(clean.ch_names)} channels '
                  f'→ {folder / (fid + "_clean-epo.fif")}')
            if dropped:
                print(f'  dropped channels: {dropped}')

            # --- decision report (non-fatal) ---
            try:
                stage_order = L.ordered_present_stages(P['stages'][insc], S['custom_stages'])
                table_html = L.build_rejection_table_html(meta, S['methods_sel'], stage_order,
                                                          title='Rejection by stage × method (final decision)',
                                                          reject_flag=fr)
                html = L.manual_decision_html(fid, decision, table_html)
                fig = L.plot_review_strip(P, fr, S['overridden'], custom_stages=S['custom_stages'],
                                          in_scope=insc)
                L.save_report_html(reports_folder / f'{fid}_manualreject_report.html',
                                   f'{fid} — manual epoch rejection', [('Final decision', fig)], html)
                print(f'  report  → {reports_folder / (fid + "_manualreject_report.html")}')
            except Exception as rep_e:
                print(f'  ⚠ Report failed ({rep_e}) — the data files above were written.')

            # --- global summary, rebuilt from every decision TSV on disk (non-fatal) ---
            try:
                summary = _rebuild_global_summary(reports_root)
                if not summary.empty:
                    summary = summary.sort_values('file_id')
                    reports_root.mkdir(parents=True, exist_ok=True)
                    summary.to_csv(reports_root / 'global_manualreject_summary.tsv', sep='\t', index=False)
                    print(f'  global summary → {reports_root / "global_manualreject_summary.tsv"} '
                          f'({len(summary)} participant(s) reviewed)')
                    cols = [c for c in ['file_id', 'n_in_scope', 'n_rejected', 'pct_rejected',
                                        'n_overrides', 'stages_used', 'methods_used'] if c in summary.columns]
                    display(HTML(summary[cols].to_html(index=False)))
            except Exception as sum_e:
                print(f'  ⚠ Global summary failed ({sum_e}).')
        except Exception as e:
            print(f'Save error: {e}')

btn_review.on_click(_on_review)
btn_save.on_click(_on_save)
display(widgets.VBox([widgets.HBox([btn_review, btn_save]), out_review]))

